# R6 — Star Schema & Metric View

Star schema over `hop.gold.orders_enriched`: `hop.analytics` mein dimensions (`dim_date`, `dim_customer`, `dim_zone`, `dim_restaurant` SCD2 versions) + `fact_orders` (liquid-clustered on `placed_date_key`), declared PK/FK constraints, aur ek metric view jahan saare business metrics (order count, delivered, cancellation rate, avg delivery minutes, late rate, gmv) sirf ek jagah defined hain — by delivery class.

In [0]:
# ============================================================
# hop.analytics.dim_date — 2026-06-01 se 2026-08-31 tak har date
# ============================================================

spark.sql("""
    CREATE OR REPLACE TABLE hop.analytics.dim_date AS
    SELECT
        CAST(DATE_FORMAT(d, 'yyyyMMdd') AS INT) AS date_key,
        d                                      AS full_date,
        CAST(YEAR(d)  AS INT)                  AS year,
        CAST(MONTH(d) AS INT)                  AS month,
        CAST(DATE_TRUNC('month', d) AS DATE)   AS month_start,
        CAST(DATE_TRUNC('week',  d) AS DATE)   AS week_start,
        DATE_FORMAT(d, 'EEEE')                 AS day_of_week
    FROM (SELECT EXPLODE(SEQUENCE(DATE'2026-06-01', DATE'2026-08-31', INTERVAL 1 DAY)) AS d)
""")

print("dim_date rows:", spark.table("hop.analytics.dim_date").count())
display(spark.table("hop.analytics.dim_date").limit(5))

In [0]:
# ============================================================
# dim_customer — NO personal data (name/email/phone excluded)
# dim_zone    — zone reference
# Surrogate keys: stable row_number by natural key
# ============================================================

spark.sql("""
    CREATE OR REPLACE TABLE hop.analytics.dim_customer AS
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY customer_id) AS BIGINT) AS customer_sk,
        customer_id,
        city,
        tier
    FROM hop.silver.customers
""")

spark.sql("""
    CREATE OR REPLACE TABLE hop.analytics.dim_zone AS
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY zone_id) AS BIGINT) AS zone_sk,
        zone_id,
        city,
        zone_type
    FROM hop.silver.zones
""")

print("dim_customer:", spark.table("hop.analytics.dim_customer").count(),
      "| dim_zone:", spark.table("hop.analytics.dim_zone").count())
display(spark.table("hop.analytics.dim_customer").limit(3))

In [0]:
# ============================================================
# dim_restaurant — one row per restaurant VERSION (SCD2)
# restaurant_sk = version ka surrogate key
# (fact is sk ko point karega jo placed ke waqt in force thi)
# ============================================================

spark.sql("""
    CREATE OR REPLACE TABLE hop.analytics.dim_restaurant AS
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY restaurant_id, START_AT) AS BIGINT) AS restaurant_sk,
        restaurant_id,
        name,
        cuisine,
        zone_id,
        rating,
        is_active,
        START_AT AS valid_from,
        END_AT   AS valid_to,
        (END_AT IS NULL) AS is_current
    FROM hop.silver.dim_restaurant_scd2
""")

print("dim_restaurant rows:", spark.table("hop.analytics.dim_restaurant").count())
display(spark.table("hop.analytics.dim_restaurant").orderBy("restaurant_id", "valid_from").limit(5))

In [0]:
# ============================================================
# fact_orders — gold.orders_enriched ka ek row per order
# - restaurant_sk: placed_ts par in-force restaurant version
# - placed_date_key: yyyymmdd INT (date filters ke liye)
# - LIQUID CLUSTERED on placed_date_key
# ============================================================

spark.sql("""
    CREATE OR REPLACE TABLE hop.analytics.fact_orders
    CLUSTER BY (placed_date_key)
    AS
    SELECT
        o.order_id,
        dc.customer_sk,
        dr.restaurant_sk,
        dz.zone_sk,
        CAST(DATE_FORMAT(o.placed_ts, 'yyyyMMdd') AS INT) AS placed_date_key,
        o.delivery_class,
        o.status,
        o.order_value,
        o.promised_minutes,
        o.actual_minutes,
        o.minutes_past_promise
    FROM hop.gold.orders_enriched o
    JOIN hop.analytics.dim_customer dc
      ON o.customer_id = dc.customer_id
    JOIN hop.analytics.dim_zone dz
      ON o.delivery_zone_id = dz.zone_id
    JOIN hop.analytics.dim_restaurant dr
      ON o.restaurant_id = dr.restaurant_id
     AND o.placed_ts >= dr.valid_from
     AND (o.placed_ts < dr.valid_to OR dr.valid_to IS NULL)
""")

gold_n = spark.table("hop.gold.orders_enriched").count()
fact_n = spark.table("hop.analytics.fact_orders").count()
print(f"gold.orders_enriched: {gold_n} | fact_orders: {fact_n} -> "
      + ("MATCH (ek row per order)" if gold_n == fact_n else "MISMATCH! joins check karo"))
display(spark.table("hop.analytics.fact_orders").limit(5))

In [0]:
# ============================================================
# Declared keys & relationships — PKs dims/fact par,
# FKs fact -> dims (informational, Unity Catalog mein declared)
# ============================================================

statements = [
    # --- Pehle key columns NOT NULL (PK ke liye zaroori, CTAS nullable banata hai)
    "ALTER TABLE hop.analytics.dim_date        ALTER COLUMN date_key         SET NOT NULL",
    "ALTER TABLE hop.analytics.dim_customer     ALTER COLUMN customer_sk     SET NOT NULL",
    "ALTER TABLE hop.analytics.dim_zone         ALTER COLUMN zone_sk         SET NOT NULL",
    "ALTER TABLE hop.analytics.dim_restaurant   ALTER COLUMN restaurant_sk   SET NOT NULL",
    "ALTER TABLE hop.analytics.fact_orders      ALTER COLUMN order_id        SET NOT NULL",
    "ALTER TABLE hop.analytics.fact_orders      ALTER COLUMN customer_sk     SET NOT NULL",
    "ALTER TABLE hop.analytics.fact_orders      ALTER COLUMN restaurant_sk   SET NOT NULL",
    "ALTER TABLE hop.analytics.fact_orders      ALTER COLUMN zone_sk         SET NOT NULL",
    "ALTER TABLE hop.analytics.fact_orders      ALTER COLUMN placed_date_key SET NOT NULL",
    # --- Primary keys
    "ALTER TABLE hop.analytics.dim_date        ADD CONSTRAINT dim_date_pk        PRIMARY KEY (date_key)",
    "ALTER TABLE hop.analytics.dim_customer     ADD CONSTRAINT dim_customer_pk   PRIMARY KEY (customer_sk)",
    "ALTER TABLE hop.analytics.dim_zone         ADD CONSTRAINT dim_zone_pk       PRIMARY KEY (zone_sk)",
    "ALTER TABLE hop.analytics.dim_restaurant   ADD CONSTRAINT dim_restaurant_pk PRIMARY KEY (restaurant_sk)",
    "ALTER TABLE hop.analytics.fact_orders      ADD CONSTRAINT fact_orders_pk     PRIMARY KEY (order_id)",
    # --- Foreign keys (fact -> dims)
    "ALTER TABLE hop.analytics.fact_orders      ADD CONSTRAINT fact_orders_customer_fk   FOREIGN KEY (customer_sk)     REFERENCES hop.analytics.dim_customer (customer_sk)",
    "ALTER TABLE hop.analytics.fact_orders      ADD CONSTRAINT fact_orders_restaurant_fk FOREIGN KEY (restaurant_sk)   REFERENCES hop.analytics.dim_restaurant (restaurant_sk)",
    "ALTER TABLE hop.analytics.fact_orders      ADD CONSTRAINT fact_orders_zone_fk        FOREIGN KEY (zone_sk)          REFERENCES hop.analytics.dim_zone (zone_sk)",
    "ALTER TABLE hop.analytics.fact_orders      ADD CONSTRAINT fact_orders_date_fk         FOREIGN KEY (placed_date_key)  REFERENCES hop.analytics.dim_date (date_key)",
]

for stmt in statements:
    name = (stmt.split("ADD CONSTRAINT ")[1].split(" ")[0] if "ADD CONSTRAINT" in stmt
            else stmt.split("ALTER COLUMN ")[1].split(" ")[0])
    try:
        spark.sql(stmt)
        print("OK  :", name)
    except Exception as e:
        print("SKIP:", name, "->", str(e).splitlines()[0])

In [0]:
# ============================================================
# hop.analytics.mv_orders — metric view over fact_orders
# joined to the dimensions (customer, zone, restaurant, date)
# gmv / avg_delivery_minutes / late_rate: delivered orders par
# ============================================================

spark.sql("""
    CREATE OR REPLACE VIEW hop.analytics.mv_orders
    WITH METRICS
    LANGUAGE YAML
    AS $$
        version: 1.1
        source: hop.analytics.fact_orders
        comment: HungryHop order business metrics by delivery class
        joins:
          - name: dim_customer
            source: hop.analytics.dim_customer
            on: source.customer_sk = dim_customer.customer_sk
            cardinality: many_to_one
            rely: {at_most_one_match: true}
          - name: dim_zone
            source: hop.analytics.dim_zone
            on: source.zone_sk = dim_zone.zone_sk
            cardinality: many_to_one
            rely: {at_most_one_match: true}
          - name: dim_restaurant
            source: hop.analytics.dim_restaurant
            on: source.restaurant_sk = dim_restaurant.restaurant_sk
            cardinality: many_to_one
            rely: {at_most_one_match: true}
          - name: dim_date
            source: hop.analytics.dim_date
            on: source.placed_date_key = dim_date.date_key
            cardinality: many_to_one
            rely: {at_most_one_match: true}
        dimensions:
          - name: delivery_class
            expr: delivery_class
        measures:
          - name: order_count
            expr: COUNT(*)
          - name: delivered_orders
            expr: COUNT(*) FILTER (WHERE status = 'delivered')
          - name: cancellation_rate
            expr: CAST(SUM(CASE WHEN status = 'cancelled' THEN 1.0 ELSE 0.0 END) / COUNT(*) AS DOUBLE)
          - name: avg_delivery_minutes
            expr: AVG(actual_minutes) FILTER (WHERE status = 'delivered')
          - name: late_rate
            expr: |-
              CAST(SUM(CASE WHEN status = 'delivered'
                        AND minutes_past_promise > CASE delivery_class WHEN 'instant' THEN 5 WHEN 'standard' THEN 20 END
                      THEN 1.0 ELSE 0.0 END)
              / NULLIF(COUNT(*) FILTER (WHERE status = 'delivered'), 0) AS DOUBLE)
          - name: gmv
            expr: SUM(order_value) FILTER (WHERE status = 'delivered')
    $$
""")

print("mv_orders created:")
display(spark.sql("DESCRIBE TABLE hop.analytics.mv_orders"))

In [0]:
# ============================================================
# VERIFY — counts, one-row-per-order, constraints,
# metric view vs direct SQL, date filter
# ============================================================

# 1) Row counts
for t in ["hop.analytics.dim_date", "hop.analytics.dim_customer", "hop.analytics.dim_zone",
          "hop.analytics.dim_restaurant", "hop.analytics.fact_orders", "hop.gold.orders_enriched"]:
    print(f"{t}: {spark.table(t).count()}")

# 2) One row per order
spark.sql("""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT order_id) AS distinct_orders
    FROM hop.analytics.fact_orders
""").show()

# 3) Declared constraints (PK/FK metadata mein hain?)
sc = spark.sql("SHOW CREATE TABLE hop.analytics.fact_orders").first()[0]
print("Declared keys:")
for l in sc.split("\n"):
    if "PRIMARY KEY" in l.upper() or "FOREIGN KEY" in l.upper():
        print(" ", l.strip())

# 4) Metric view — by delivery class
print("\nMetric view (by delivery class):")
display(spark.sql("""
    SELECT `delivery_class`,
           MEASURE(`order_count`)         AS order_count,
           MEASURE(`delivered_orders`)    AS delivered_orders,
           MEASURE(`cancellation_rate`)   AS cancellation_rate,
           MEASURE(`avg_delivery_minutes`) AS avg_delivery_minutes,
           MEASURE(`late_rate`)           AS late_rate,
           MEASURE(`gmv`)                 AS gmv
    FROM hop.analytics.mv_orders
    GROUP BY ALL
    ORDER BY `delivery_class`
"""))

# 5) Cross-check — direct SQL se same numbers
print("Direct SQL cross-check (values match honi chahiye):")
display(spark.sql("""
    SELECT delivery_class,
           COUNT(*) AS order_count,
           COUNT(*) FILTER (WHERE status = 'delivered') AS delivered_orders,
           SUM(CASE WHEN status = 'cancelled' THEN 1.0 ELSE 0.0 END) / COUNT(*) AS cancellation_rate,
           AVG(actual_minutes) FILTER (WHERE status = 'delivered') AS avg_delivery_minutes,
           SUM(CASE WHEN status = 'delivered'
                    AND minutes_past_promise > CASE delivery_class WHEN 'instant' THEN 5 WHEN 'standard' THEN 20 END
                  THEN 1.0 ELSE 0.0 END)
             / NULLIF(COUNT(*) FILTER (WHERE status = 'delivered'), 0) AS late_rate,
           SUM(order_value) FILTER (WHERE status = 'delivered') AS gmv
    FROM hop.analytics.fact_orders
    GROUP BY delivery_class
    ORDER BY delivery_class
"""))

# 6) Date filter suit check (placed_date_key clustered column)
print("June 2026 orders (placed_date_key filter):")
display(spark.sql("""
    SELECT COUNT(*) AS june_orders
    FROM hop.analytics.fact_orders
    WHERE placed_date_key BETWEEN 20260601 AND 20260630
"""))

In [0]:
# ============================================================
# METRIC FALLBACK — definitions table + plain SQL view
# (metric view mv_orders ke bina bhi wahi meanings)
# ============================================================

# 1) hop.analytics.metric_definitions (metric_name, expression, description)
# NOTE: SQL string-escaping quotes kha jaata hai, isliye Python se likhte hain
metrics = [
    ("order_count",
     "COUNT(*)",
     "Total orders placed (all statuses)"),
    ("delivered_orders",
     "COUNT(*) FILTER (WHERE status = 'delivered')",
     "Orders that were delivered"),
    ("cancellation_rate",
     "CAST(SUM(CASE WHEN status = 'cancelled' THEN 1.0 ELSE 0.0 END) / COUNT(*) AS DOUBLE)",
     "Cancelled orders divided by all orders"),
    ("avg_delivery_minutes",
     "AVG(actual_minutes) FILTER (WHERE status = 'delivered')",
     "Average actual minutes of delivered orders"),
    ("late_rate",
     "CAST(SUM(CASE WHEN status = 'delivered' AND minutes_past_promise > CASE delivery_class WHEN 'instant' THEN 5 WHEN 'standard' THEN 20 END THEN 1.0 ELSE 0.0 END) / NULLIF(COUNT(*) FILTER (WHERE status = 'delivered'), 0) AS DOUBLE)",
     "Late delivered orders (minutes_past_promise > 5 for instant, > 20 for standard) divided by delivered orders; cancelled orders are never late"),
    ("gmv",
     "SUM(order_value) FILTER (WHERE status = 'delivered')",
     "Total order_value of delivered orders, in rupees"),
]

(spark.createDataFrame(metrics, "metric_name STRING, expression STRING, description STRING")
     .write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable("hop.analytics.metric_definitions"))

# 2) hop.analytics.v_metrics_by_class — plain view, mv_orders jaisi hi values
spark.sql("""
    CREATE OR REPLACE VIEW hop.analytics.v_metrics_by_class AS
    SELECT
        delivery_class,
        COUNT(*) AS order_count,
        COUNT(*) FILTER (WHERE status = 'delivered') AS delivered_orders,
        CAST(SUM(CASE WHEN status = 'cancelled' THEN 1.0 ELSE 0.0 END) / COUNT(*) AS DOUBLE) AS cancellation_rate,
        AVG(actual_minutes) FILTER (WHERE status = 'delivered') AS avg_delivery_minutes,
        CAST(SUM(CASE WHEN status = 'delivered'
                      AND minutes_past_promise > CASE delivery_class WHEN 'instant' THEN 5 WHEN 'standard' THEN 20 END
                    THEN 1.0 ELSE 0.0 END)
             / NULLIF(COUNT(*) FILTER (WHERE status = 'delivered'), 0) AS DOUBLE) AS late_rate,
        SUM(order_value) FILTER (WHERE status = 'delivered') AS gmv
    FROM hop.analytics.fact_orders
    GROUP BY delivery_class
""")

# ============================================================
# VERIFY — mv_orders vs v_metrics_by_class same hona chahiye
# ============================================================

mv = spark.sql("""
    SELECT `delivery_class`,
           MEASURE(`order_count`)          AS order_count,
           MEASURE(`delivered_orders`)     AS delivered_orders,
           MEASURE(`cancellation_rate`)    AS cancellation_rate,
           MEASURE(`avg_delivery_minutes`) AS avg_delivery_minutes,
           MEASURE(`late_rate`)            AS late_rate,
           MEASURE(`gmv`)                  AS gmv
    FROM hop.analytics.mv_orders
    GROUP BY ALL
""")
vb = spark.table("hop.analytics.v_metrics_by_class")

d1 = mv.exceptAll(vb).count()
d2 = vb.exceptAll(mv).count()
print(f"mv_orders vs v_metrics_by_class EXCEPT (dono taraf): {d1}, {d2} ->"
      + (" MATCH — same meanings" if d1 == 0 and d2 == 0 else " MISMATCH!"))

print("\nmetric_definitions:")
display(spark.table("hop.analytics.metric_definitions"))
print("\nv_metrics_by_class:")
display(spark.table("hop.analytics.v_metrics_by_class"))

In [0]:
%sql
SELECT `delivery_class`,
       MEASURE(`order_count`)          AS order_count,
       MEASURE(`delivered_orders`)     AS delivered_orders,
       MEASURE(`cancellation_rate`)    AS cancellation_rate,
       MEASURE(`avg_delivery_minutes`) AS avg_delivery_minutes,
       MEASURE(`late_rate`)            AS late_rate,
       MEASURE(`gmv`)                  AS gmv
FROM hop.analytics.mv_orders
GROUP BY ALL
ORDER BY `delivery_class`; 